In [6]:
import os
import cv2
import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchvision import transforms

# Constants
SAMPLE_RATE = 16000
FRAME_SIZE = 112  # Reduced for CPU efficiency
NUM_MFCC = 20
EMOTIONS = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry', '06': 'fearful', '07': 'disgust', '08': 'surprised'
}
DATASET_PATH = "data"  # Update with your actual path
BATCH_SIZE = 8
NUM_EPOCHS = 10

# Custom Dataset for RAVDESS
class RAVDESSDataset(Dataset):
    def __init__(self, file_list, transform=None):
        self.file_list = file_list
        self.transform = transform
        self.emotions = EMOTIONS

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        file_path = self.file_list[idx]
        # Parse filename to get emotion label
        filename = os.path.basename(file_path)
        emotion_code = filename.split('-')[2]
        label = list(self.emotions.keys()).index(emotion_code)

        # Process audio
        audio_path = file_path.replace('.mp4', '.wav') if file_path.endswith('.mp4') else file_path
        if not os.path.exists(audio_path):
            audio_path = file_path  # Fallback to mp4 if wav not found
        audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE)
        mfcc = librosa.feature.mfcc(y=audio, sr=SAMPLE_RATE, n_mfcc=NUM_MFCC)
        mfcc = np.mean(mfcc.T, axis=0)  # Average over time

        # Process video (use dummy frame if no video)
        if file_path.endswith('.mp4'):
            cap = cv2.VideoCapture(file_path)
            frames = []
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret:
                    break
                frame = cv2.resize(frame, (FRAME_SIZE, FRAME_SIZE))
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(frame)
            cap.release()

            # Select middle frame
            if len(frames) > 0:
                frame = frames[len(frames)//2]
            else:
                frame = np.zeros((FRAME_SIZE, FRAME_SIZE, 3), dtype=np.uint8)
        else:
            frame = np.zeros((FRAME_SIZE, FRAME_SIZE, 3), dtype=np.uint8)  # Dummy frame for audio-only

        if self.transform:
            frame = self.transform(frame)

        return torch.FloatTensor(mfcc), frame, label

# Audio CNN Model
class AudioCNN(nn.Module):
    def __init__(self, num_classes):
        super(AudioCNN, self).__init__()
        self.conv1 = nn.Conv1d(NUM_MFCC, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(32, 128)  # Corrected to match flattened conv output
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = x.unsqueeze(2)  # Add channel dimension [batch, NUM_MFCC, 1]
        x = self.relu(self.bn1(self.conv1(x)))  # [batch, 32, 1]
        x = x.view(x.size(0), -1)  # Flatten to [batch, 32]
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Video CNN Model
class VideoCNN(nn.Module):
    def __init__(self, num_classes):
        super(VideoCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(32 * (FRAME_SIZE // 4) * (FRAME_SIZE // 4), 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Multimodal Fusion Model
class MultimodalModel(nn.Module):
    def __init__(self, num_classes):
        super(MultimodalModel, self).__init__()
        self.audio_cnn = AudioCNN(num_classes)
        self.video_cnn = VideoCNN(num_classes)
        self.fc = nn.Linear(num_classes * 2, num_classes)

    def forward(self, audio, video):
        audio_out = self.audio_cnn(audio)
        video_out = self.video_cnn(video)
        combined = torch.cat((audio_out, video_out), dim=1)
        out = self.fc(combined)
        return out

# Data Preparation
def prepare_data(dataset_path):
    if not os.path.exists(dataset_path):
        raise FileNotFoundError(f"Dataset path '{dataset_path}' does not exist. Please update DATASET_PATH.")

    file_list = []
    for root, _, files in os.walk(dataset_path):
        for file in files:
            if file.endswith(('.mp4', '.wav')):  # Support both mp4 and wav
                file_list.append(os.path.join(root, file))
    
    if not file_list:
        raise ValueError(f"No .mp4 or .wav files found in '{dataset_path}'. Please check the dataset structure and DATASET_PATH.")

    # Use a smaller subset for CPU (e.g., 30% of data)
    file_list, _ = train_test_split(file_list, test_size=0.7, random_state=42)
    
    train_files, test_files = train_test_split(file_list, test_size=0.2, random_state=42)
    train_files, val_files = train_test_split(train_files, test_size=0.25, random_state=42)
    
    return train_files, val_files, test_files

# Training Function
def train_model(model, train_loader, val_loader):
    device = torch.device("cpu")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0
        for audio, video, labels in train_loader:
            audio, video, labels = audio.to(device), video.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(audio, video)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for audio, video, labels in val_loader:
                audio, video, labels = audio.to(device), video.to(device), labels.to(device)
                outputs = model(audio, video)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        print(f'Epoch {epoch+1}, Train Loss: {running_loss/len(train_loader):.4f}, '
              f'Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {100 * correct/total:.2f}%')

# Main Execution
if __name__ == "__main__":
    # Data transforms with augmentation for video frames
    transform = transforms.Compose([
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(p=0.5),  # Data augmentation
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Prepare datasets
    try:
        train_files, val_files, test_files = prepare_data(DATASET_PATH)
    except (FileNotFoundError, ValueError) as e:
        print(e)
        exit(1)
    
    train_dataset = RAVDESSDataset(train_files, transform=transform)
    val_dataset = RAVDESSDataset(val_files, transform=transform)
    test_dataset = RAVDESSDataset(test_files, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    # Initialize and train model
    model = MultimodalModel(num_classes=len(EMOTIONS))
    train_model(model, train_loader, val_loader)

    # Test model
    device = torch.device("cpu")
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for audio, video, labels in test_loader:
            audio, video, labels = audio.to(device), video.to(device), labels.to(device)
            outputs = model(audio, video)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Test Accuracy: {100 * correct/total:.2f}%')

Epoch 1, Train Loss: 2.1558, Val Loss: 1.8391, Val Acc: 28.90%
Epoch 2, Train Loss: 1.8923, Val Loss: 1.8002, Val Acc: 29.48%
Epoch 3, Train Loss: 1.8367, Val Loss: 1.7466, Val Acc: 31.79%
Epoch 4, Train Loss: 1.8221, Val Loss: 1.7106, Val Acc: 31.21%
Epoch 5, Train Loss: 1.7679, Val Loss: 1.6900, Val Acc: 31.79%
Epoch 6, Train Loss: 1.7263, Val Loss: 1.6837, Val Acc: 34.10%
Epoch 7, Train Loss: 1.6956, Val Loss: 1.7031, Val Acc: 30.06%
Epoch 8, Train Loss: 1.6386, Val Loss: 1.6221, Val Acc: 36.99%
Epoch 9, Train Loss: 1.6457, Val Loss: 1.6491, Val Acc: 34.68%
Epoch 10, Train Loss: 1.5815, Val Loss: 1.7469, Val Acc: 33.53%
Test Accuracy: 30.64%
